# Project OverView

The project was developed for Enbridge, a leading Canadian energy infrastructure company with a significant presence in the renewable energy sector. The objective was to build an AI-powered assistant for wind turbine field engineers responsible for inspecting, maintaining, and repairing wind turbines.

The assistant enabled engineers to quickly access critical information such as maintenance manuals, troubleshooting guides, standard operating procedures (SOPs), safety regulations, and historical maintenance records through natural language queries. By leveraging Generative AI and Retrieval-Augmented Generation (RAG), the solution delivered accurate, context-aware responses from the organization's knowledge base, reducing the need to manually search through large volumes of technical documentation.

This improved field productivity, reduced downtime, accelerated issue resolution, and helped engineers make informed maintenance decisions while ensuring compliance with safety standards.



# RAG

## Indexing Stage

### Azure Blog Storage

All company documents (manuals, reports, diagrams, and safety guidelines) are stored in Microsoft Azure Blob Storage, which provides centralized, scalable, and secure storage

### Azure Function

Whenever a new document is uploaded to Azure Blob Storage, an Azure Function is automatically triggered. This function checks the document and adds a task to an Azure Storage Queue. Another Azure Function reads the task from the queue, opens the document, extracts the content, and sends it to the AI pipeline.

### Azure Document Intelligence

Azure Document Intelligence extracts content from PDFs, scanned files, and images. It identifies document structure (headings, paragraphs, tables, lists) and uses OCR(Optical Character Recognition )  to extract text from scanned documents and diagrams.

The output is converted into structured Markdown instead of plain text, preserving headers and document hierarchy for better chunking.

### PyMuPDF 

If the PDF contains images or diagrams, we use PyMuPDF to extract them and save them separately in Azure Blob Storage. We also store metadata, such as the image name and page number, so each image remains linked to the relevant text.

When a user asks a question, the RAG system retrieves both the relevant text and the associated image references. FastAPI then generates a secure SAS URL so the images can be displayed along with the answer, providing a multimodal response.

### Chucking Strategy 

First, the extracted Markdown content is split based on document headings using LangChain's MarkdownHeaderTextSplitter. This preserves the document's logical structure, ensuring that related sections such as maintenance procedures, safety instructions, and alarm descriptions remain together.

Next, the chunks are optimized using token-based chunking. Smaller sections are merged until they reach the desired chunk size, while larger sections are further divided using RecursiveCharacterTextSplitter to stay within the LLM's token limits.

To preserve context, the system applies chunk overlap, where a small portion of the previous chunk is repeated in the next chunk. This helps maintain continuity when information spans multiple chunks.

After the chunks are created, Azure OpenAI extracts structured metadata for each chunk, such as the procedure name, procedure code, alarm code, related figures or tables, and a short summary.

Finally, each chunk, along with its metadata, is stored in a structured JSON format. 

### Azure OpenAI Embedding

In this step, each document chunk is converted into a vector embedding using text-embedding-ada-002. The embedding is a 1536-dimensional vector that captures the semantic meaning of the text.

This allows the system to perform semantic search, where similar meanings can be matched even if the words are different. All document chunks are converted into embeddings and stored.

### Azure AI Search Index

After generating embeddings, we store each document chunk in Azure AI Search along with its metadata and embedding vector. Azure AI Search indexes both the text and the vector, enabling keyword search as well as semantic search. This helps retrieve the most relevant document chunks for the user's query.

## Retrieval Stage

### authentication and authorization 
Every request first goes through authentication to verify the user's identity and authorization to check their permissions.

### Azure cache Redis Rate Limit

We use Azure Azure cache Redis for rate limiting.It+ checks the number of requests a user sends (for example, 10 requests per minute). If the limit is exceeded, it returns an HTTP 429 (Too Many Requests) response without forwarding the request to FastAPI or Azure OpenAI, protecting the application from abuse and reducing unnecessary LLM costs.

### Azure cache Redis (token limit )

We use Azure Cache for Redis to track each user's daily token usage and block requests that exceed the configured limit before they reach the LLM, helping control costs and prevent misuse.

### Azure AI Language 

Azure AI Language is mainly used for PII detection. It identifies sensitive information such as engineer names, email IDs, employee IDs, and other personal data in user queries or documents. We redact this information before sending the request to the LLM, while preserving technical information like turbine IDs, alarm codes, and component names because they are required for accurate retrieval.

### Azure AI Content Safety guardrails

We apply Azure AI Content Safety guardrails at both input and output stages. Before retrieval, we validate and filter user queries to prevent harmful inputs and prompt injection attacks. After Azure OpenAI generates the response, we apply output guardrails to detect sensitive information, harmful content, or policy violations before sending the answer back to the user.


### Pydantic and Regex Validation 

We first use Pydantic to validate the request schema. It checks that all required fields are present. After that, we use Regex validation to verify specific input patterns, such as conversation IDs or allowed characters, and to sanitize the input if necessary. Together, these validations ensure that only valid and well-formed requests enter the RAG pipeline.

### Query rewrite 

we determine whether the query is a new question or a follow-up. If it's a follow-up, we retrieve the relevant conversation history from Azure Cosmos DB. We then combine the current query with the conversation context and use an LLM to rewrite it into a clear, standalone question. The rewritten query is converted into an embedding and sent to Azure AI Search

### Azure AI Search 

In this stage, the system performs hybrid retrieval using Azure AI Search to find the most relevant document chunks.

First, keyword search is performed using BM25, which retrieves chunks containing exact matches of the query terms. This is especially useful for technical queries involving error codes, component IDs, or specific terminology.

At the same time, the query is converted into an embedding, and vector search is performed using KNN (K-Nearest Neighbors). The system retrieves the Top-K nearest document vectors based on cosine similarity, allowing it to find semantically similar content even when different words are used.

The results from both keyword and vector search are then combined using RRF (Reciprocal Rank Fusion), which produces a single ranked list of relevant document chunks.

Next, Semantic Ranker is applied to re-score these retrieved chunks based on their relevance to the user's query. This improves retrieval accuracy by promoting the most relevant chunks to the top.

The final Top-K high-quality chunks are then passed to Azure OpenAI to generate an accurate response.


## Generative Stage

### LLM response 

After retrieval, the final Top-K relevant document chunks are combined with the user's query, conversation history, and a carefully designed system prompt. This complete prompt is sent to Azure OpenAI (GPT-4o). The LLM generates a grounded response using only the retrieved context instead of relying on its own knowledge. The system prompt instructs the model to answer only from the provided documents, avoid hallucinations, and clearly state when sufficient information is not available.

### Azure AI content Safety

the FastAPI backend performs post-processing before returning it to the user. First, Azure AI Content Safety validates the generated output to ensure it does not contain harmful or policy-violating content. 

### Pydantic scheme validation and regrex

we first validate the structured JSON using a Pydantic schema to ensure all required fields and data types are correct. We then apply regex-based sanitization to clean any unwanted formatting or extra text. Finally, returning the playload response to the frontend.


# Security

## Authentication

Our application uses Microsoft Entra ID in a single-tenant configuration, so only employees from our organization can access the AI Assistant. During authentication, the React frontend redirects unauthenticated users to Microsoft Entra ID, where they sign in using their corporate credentials and complete MFA if required. After successful authentication, Microsoft Entra ID issues an ID token for the frontend and an access token (JWT) for the backend. Every API request includes the JWT, and the FastAPI backend validates its signature, issuer, audience, tenant ID, and expiration time.

## Authorization
After authentication, the backend performs authorization using Role-Based Access Control (RBAC). It extracts the user's role or Azure AD group from the JWT and checks whether the user has permission to perform the requested operation. For example, engineers can query the AI Assistant, while administrators can also upload documents and manage the search index. If the user is authorized, the request enters the AI pipeline. If authorization fails, the backend immediately returns a 403 Forbidden response.

## Azure AI Language 

PII Detection identifies personally identifiable information such as names, email addresses, phone numbers, passport numbers, and credit card numbers. It returns the detected entities along with their categories and confidence scores. PII Redaction builds on detection by masking or replacing the sensitive information before it is stored, logged, or sent to an LLM. Detection is used for analysis, while redaction is used for privacy protection.

## Azure AI Content Safety 

is a service that detects, filters, and blocks harmful, unsafe, or policy-violating content in AI applications. It helps ensure that both user inputs and LLM-generated outputs comply with safety policies. It also helps detect prompt injection and jailbreak attempts.


Prompt Injection is an attack where malicious instructions are inserted into user prompts or external content, such as RAG documents, to manipulate the application's behavior. 
Jailbreak is an attempt to bypass the LLM's built-in safety policies and generate responses that would normally be restricted.

# Azure Cache for Redis 

Azure Cache for Redis is used as a high-performance in-memory datastore to store temporary data that needs very fast access. It improves application performance, reduces Azure OpenAI costs, and enables state sharing across multiple FastAPI instances.

## Token Budget 

tracks and limits each user's daily LLM token usage by storing the consumed tokens in Redis using a date-based key. After every successful LLM request, it atomically updates the token count using INCRBY to ensure accuracy even with concurrent requests. Before processing a new request, it checks whether the user has exceeded the daily token limit. The Redis key is configured with a 24-hour TTL, so it automatically expires and resets the user's daily token budget without requiring manual cleanup."

## API Rate Limit 

We use a Redis Sorted Set–based sliding-window rate limiter to track requests per user, enforce request limits (10 RPM) within a time window, and return HTTP 429 when the rate limit is exceeded.

### TTL (Time To Live): 

A Redis expiration timer that automatically deletes the daily token key at midnight, resetting the user's token budget for the next day.

## chat history  

redis_cache_service.py
 that handle storing and retrieving chat histories capped at 5 messages by default:

add_chat_message: Appends a new chat message to a Redis list using RPUSH and then uses LTRIM to keep only the last max_history items (defaults to 5). It also refreshes the TTL of the chat history so it eventually expires according to your configuration.
get_chat_history: Retrieves the entire list for a given session using LRANGE.

## Rag cache 

We use Redis as a cache to avoid searching the same documents again for repeated questions.

When a user asks a question, we create a unique hash key from the question.     
We first check Redis to see if the document context is already cached.          
Cache Hit: We get the context directly from Redis and skip the search, making the response much faster.         
Cache Miss: We perform the normal RAG process (embedding + Azure AI Search), get the relevant documents, and then store the context in Redis.       
The cached data has a TTL (Time To Live) of 1 hour, so Redis automatically removes old data and the system uses updated documents.      
If Redis is unavailable, the application doesn't fail. It simply skips the cache and continues with the normal RAG flow, ensuring high availability.        

## rate_limiter_service.py

We store the user_id (from the authenticated user token) in Redis for two main purposes:

Rate Limiting: A key like ratelimit:{user_id} tracks how many requests the user makes in a short time (for example, 60 requests per minute) to prevent abuse.       
Token Budgeting: A key like tokenlimit:{user_id}:2026-07-29 stores the number of LLM tokens the user has used that day, ensuring they don't exceed their daily limit (for example, 100,000 tokens/day).

## safety

We use SCAN instead of KEYS because KEYS can slow down Redis by checking all keys at once. SCAN checks keys little by little, so Redis keeps working normally. We delete the matching keys, count how many were deleted, and log the result. If any Redis error happens, we catch it, log the error, and prevent the application from crashing.

# Observability Metrics

We use **Azure Monitor + Application Insights** to continuously monitor the health, performance, reliability, and cost of our RAG chatbot. These metrics provide aggregated insights across all requests and help us create dashboards, alerts, and capacity planning.

## FastAPI Response Time (Latency)

Measures the total time taken by the backend to process a user request.

**Why it matters:** High latency indicates slower responses and impacts the overall user experience.


## Request Count

Measures the total number of API requests received by the application.

**Why it matters:** Helps monitor application traffic, usage patterns, and system load.


## Error Rate (4xx/5xx)

Measures the percentage of failed API requests.

**Why it matters:** A high error rate indicates application issues, service failures, or invalid client requests.

## Azure OpenAI Response Latency

Measures the time taken by Azure OpenAI (GPT-4o) to generate a response.

**Why it matters:** The LLM usually contributes the largest portion of the total response time, making this metric critical for performance monitoring.

## Total Token Usage

Measures the total number of prompt and completion tokens consumed.

**Why it matters:** Token usage directly impacts Azure OpenAI cost and helps optimize prompts and control daily token budgets.


## Azure AI Search Query Latency

Measures the time required to retrieve relevant document chunks from Azure AI Search.

**Why it matters:** Faster retrieval improves the overall response time of the RAG pipeline.


## Semantic Ranker Latency

Measures the time spent reranking retrieved document chunks using Semantic Ranker.

**Why it matters:** Ensures highly relevant documents are ranked first while keeping additional latency under control.


## Redis Cache Hit Ratio

Measures the percentage of requests served directly from Azure Cache for Redis.

**Why it matters:** A high cache hit ratio reduces Azure OpenAI calls, lowers cost, and improves response time.

## Cosmos DB RU Consumption

Measures the Request Units (RU) consumed by Cosmos DB operations.

**Why it matters:** Helps optimize database performance, avoid throttling, and control operational costs.


## Azure Function Execution Time

Measures the time taken by Azure Functions during document ingestion and indexing.

**Why it matters:** Slow execution delays document processing and makes newly uploaded content available later.


## End-to-End Request Duration

Measures the total time taken for a request to travel through the complete RAG pipeline—from FastAPI, Redis, Cosmos DB, Azure AI Search, and Azure OpenAI until the response is returned.

**Why it matters:** Provides an overall view of application performance and helps identify bottlenecks across multiple services.



# RAG evaluation metrics

## Context Precision

Context Precision measures how many of the retrieved documents are actually relevant to the user’s query.
It helps identify whether the system is returning useful information or unnecessary noise.
Higher context precision improves answer quality by ensuring the model focuses only on relevant context.

## Context Recall

Context Recall measures whether the system retrieves all the relevant information needed to answer a query.
It helps identify if important context is missing from the retrieved results.
Higher context recall ensures the model has enough information to generate complete and accurate answers.

## MRR (Mean Reciprocal Rank)

MRR measures how early the first relevant document appears in the ranked retrieval results.
It gives higher scores when the correct document is ranked at the top.
Higher MRR means users (and the LLM) can find useful information quickly.

## Hit rate@K

Hit Rate@K measures whether at least one relevant document appears in the top K retrieved results.
It checks if the system is able to “hit” the correct context within the first K results.
A higher Hit Rate@K means the retrieval system is more likely to provide useful information for answer generation.

## Context Relevancy
Measures how relevant the retrieved chunks are to the query.
Evaluates retrieval quality.

## Faithfulness (Groundedness)

Measures whether all claims in the generated answer are supported by the retrieved context.
Helps detect hallucinations.
High faithfulness means the model is not inventing information.

## Answer Relevancy
Checks if the generated answer directly addresses the user’s question.
Ensures the response stays on-topic.
High relevance means the answer matches the user’s intent. 

## Answer Correctness
Measures how close the generated answer is to the ground-truth answer.
Often evaluated using semantic similarity rather than exact match.
High correctness means the answer is factually accurate.

## Noise Robustness
Evaluates how well the model handles irrelevant or noisy retrieved context.
A robust system still produces correct answers despite distractions.
High robustness means better stability in real-world scenarios.




# Latency Optimization

## Azure Latency 

We reduced latency by initializing the Azure AI Content Safety and PII Detection clients once during FastAPI startup using the lifespan event, so the same connections are reused for all requests. We also execute the Content Safety and PII Detection API calls in parallel using asyncio.gather() instead of sequentially. This reduces overall response time and improves application performance.

## LLM Response Cache 

To reduce LLM latency and cost, we implement caching using Azure Cache for Redis. First, we check an exact-match cache for identical user queries. If a cached response exists, we return it immediately without calling the LLM. For semantically similar queries, we use embeddings to compare the new query with previously cached queries. If the similarity score is above a threshold, such as 0.95, we return the cached response instead of generating a new one. This significantly reduces response time, lowers Azure OpenAI token usage, and improves system throughput


## Caching History: 

Fetching chat history from Cosmos DB on every message slows things down. We could cache the recent session history in Azure Redis.

## LLM Latency 

To optimize it, we stream responses using Server-Sent Events so users receive tokens immediately, improving the Time to First Token. 
We use model routing by sending simple requests to smaller, faster models and complex requests to larger models. 

## Retrieval Latency  

We optimize Azure AI Search by applying metadata filters before vector search. By narrowing the search to relevant documents—for example, by department or document type—we reduce the search space, which improves retrieval speed and the relevance of the results

We optimize context size by retrieving multiple documents from Azure AI Search and using Azure Semantic Ranker to select only the most relevant chunks. By sending only the top 3–5 chunks to the LLM, we reduce prompt size, improve TTFT, lower token costs, and maintain answer quality

## Pdf Extraction 

### Step-by-Step

### 1. Upload PDF

* User uploads a PDF to **Azure Blob Storage**.

### 2. Extract Layout

* **Azure AI Document Intelligence** reads the PDF.
* It extracts:

  * Text
  * Headings
  * Tables
  * Page numbers
  * Table structure (rows and columns)
* Output is **Markdown**, preserving the document layout.

### 3. Identify Content

The extracted content is separated into:

* Normal text
* Tables

### 4. Merge Multi-Page Tables

If a table continues onto the next page:

* Detect same columns.
* Remove duplicate header rows.
* Merge into one complete table.

### 5. Split Large Tables

If a table has many rows:

* Split it into chunks (for example, **15 rows per chunk**).
* Add the table header to every chunk.
* Add the section heading so every chunk has context.

### 6. Clean the Data

Fix OCR mistakes such as:

* Incorrect currency symbols
* Special characters
* Formatting issues

### 7. Create Parent and Child Documents

* Store the **full table** as the **parent document**.
* Store each **15-row chunk** as a **child document**.
* Link them using a unique **parent_id**.

### 8. Enrich with Azure OpenAI

For each chunk, generate:

* Summary
* Keywords
* Sample questions (HyDE)

This improves search quality.

### 9. Generate Embeddings

Create vector embeddings using **text-embedding-3-large**.

### 10. Store in Azure AI Search

Index each chunk with:

* Text
* Embeddings
* Metadata
* Keywords
* Parent ID

Now the PDF is ready for fast hybrid retrieval.

## Azure and pyMuPDF

You use Azure because it has the "brain" (AI) to understand complex layouts, read tables, and perform OCR on scanned text. You use PyMuPDF because it has direct access to the "file system" of the PDF, allowing it to instantly and freely rip out the original, high-quality image files.

By combining them, you get the highest quality text and the highest quality images, at the fastest speed!


To handle a high load like 50,000 requests, this project relies on a highly scalable, distributed architecture designed around cloud-native Azure services and stateless backend nodes.

Here is a breakdown of how this project is structured to scale:

1. Stateless, Horizontally Scalable Backend
FastAPI & Uvicorn: The core backend is built on FastAPI, an extremely fast asynchronous Python framework. Because it's asynchronous (async/await), a single instance can handle many concurrent requests efficiently without blocking.
Horizontal Scaling: The backend instances themselves are stateless. This means you can deploy the FastAPI application to platforms like Azure Container Apps, Azure Kubernetes Service (AKS), or Azure App Service, and seamlessly scale out from 1 replica to 100+ replicas to distribute the 50,000 requests behind a load balancer.
2. Centralized State via Azure Cache for Redis
When you run multiple backend replicas, they can't store state in local memory. This project solves that by using Azure Cache for Redis for all stateful operations:

Rate Limiting: The RateLimiter (in rate_limiter_service.py) uses Redis sorted sets to track request counts across all backend instances simultaneously.
Token Budgeting: Daily token consumption is tracked in Redis.
Session/Chat History: As seen in recent updates, chat histories are stored in Redis using lists. Because Redis handles operations entirely in memory and is incredibly fast, it acts as the centralized brain allowing the web instances to scale infinitely without race conditions.
3. Highly Scalable Data Layer
Azure Cosmos DB NoSQL: The primary database is Azure Cosmos DB (azure-cosmos in requirements.txt). Cosmos DB is a globally distributed, multi-model database that offers guaranteed single-digit millisecond response times and automatic, elastic scaling of throughput and storage. It can easily handle tens of thousands of requests per second.
Azure Blob Storage: For unstructured data (files, documents, etc.), the project uses Azure Blob Storage, which is built for massive scale and high throughput.
4. Safeguards against Abuse
At a 50,000 request scale, a few bad actors can bring down a system. This project includes built-in protections:

Sliding-Window Rate Limiting: Ensures no single user can flood the system (e.g., limiting users to 60 requests/minute).
Token Budgets: Prevents unbounded cost spikes when using external LLMs (e.g., a cap of 100,000 tokens per user, per day).
Fail-Open Design: If the Redis cache temporarily goes down under extreme load, the rate limiter falls back to allowing requests (fail-open) rather than bringing down the entire application.
5. Serverless Components
You have an azure-functions/ directory containing a host.json. This indicates that parts of this architecture may be offloaded to Azure Functions. Azure Functions provide event-driven, serverless compute that can automatically scale from zero to thousands of concurrent executions instantaneously to handle massive traffic spikes without manual intervention.

In Summary:
To serve 50,000 requests (e.g., per minute), the load balancer simply distributes the traffic across many identical, stateless FastAPI containers. These containers rely on Azure Cache for Redis to coordinate rate limits and state, while pushing persistent data into Azure Cosmos DB—both of which are enterprise-grade services built specifically for planetary-scale workloads.